# Stage 00b1 — Grouping, Structured Symlinks & Batch Assignment

What this notebook does:
1. Runs grouper.run_grouping() to assign company_canonical and prototype_label to every patent
2. Assigns batch_id so that patents in the same cluster stay together, with ≥300 patents per batch
3. Writes data/batches.xlsx — one sheet per batch, plus a Summary sheet
4. Creates symlinks: structured/{company_canonical}/{prototype_label}/{patent_id}/ → matched/{patent_id}/
   (symlinks for patents not yet processed by 00b2 are silently skipped — re-run cell 4 after all batches)

Run before: 00b2 (generates batches.xlsx that 00b2 uses for batch selection)
Run after all 00b2 batches: re-run cell 4 to create symlinks for all processed patents

In [1]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import sys, importlib.util
from pathlib import Path

_cwd = Path().resolve()
repo_root = None
for _candidate in [_cwd, *_cwd.parents]:
    if (_candidate / "src").exists() and (_candidate / "config.yaml").exists():
        repo_root = _candidate
        break
if repo_root is None:
    raise RuntimeError(f"Cannot find repo root from {_cwd}.")

for p in [str(repo_root), str(repo_root / "src")]:
    while p in sys.path:
        sys.path.remove(p)
sys.path.insert(0, str(repo_root / "src"))
sys.path.insert(0, str(repo_root))
# deduplicator.py is currently archived (not part of the active src/ package).
# Add src/_archive to sys.path so cell 2's `from deduplicator import run_deduplication` resolves.
sys.path.insert(0, str(repo_root / "src" / "_archive"))

_cl_spec = importlib.util.spec_from_file_location("config_loader", repo_root / "src" / "config_loader.py")
_cl_mod  = importlib.util.module_from_spec(_cl_spec)
_cl_spec.loader.exec_module(_cl_mod)
load_config = _cl_mod.load_config

cfg = load_config()
print(f"matched    : {cfg['paths']['matched']}")
print(f"structured : {cfg['paths']['structured']}")
print(f"reviewed   : {cfg['paths']['reviewed']}")

matched    : /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/matched
structured : /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/structured
reviewed   : /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/reviewed


In [2]:
# ── Load Excel + run grouper ───────────────────────────────────────────────────
import pandas as pd
from extractor import load_patseer_excel
from deduplicator import run_deduplication
from grouper import run_grouping

raw_df = pd.read_excel(cfg["paths"]["patseer_excel"], dtype=str)
deduped_df, _ = run_deduplication(raw_df, cfg)
grouped_df    = run_grouping(deduped_df, cfg)

print(f"\nColumns available: {list(grouped_df.columns)}")
print(f"Total patents after dedup: {len(grouped_df)}")
print(f"\nTop clusters:")
print(grouped_df.groupby(["company_canonical","prototype_label"]).size().sort_values(ascending=False).head(20).to_string())

/home/vasco/anaconda3/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


[deduplicator] Columns resolved:
  pub_number  → 'Record Number'
  family_id   → 'Simple Family ID'
  filing_date → 'Filing/Application Date'
  fig_count   → None  (None = column absent, default 0)
  assignee    → 'Assignee'



[deduplicator] Summary
  Input rows          :  1,639
  Unique families     :  1,639
  Patents kept        :  1,639
  Patents dropped     :      0
  Families > 1 member :      0
  Families to review  :      0  (figure count spread > 3)


  Saved: /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/data/deduplicated_patents.csv
  Saved: /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/data/family_map.csv
[grouper] Columns resolved:
  pub_number  → 'Record Number'
  assignee    → 'Assignee'
  filing_date → 'Filing/Application Date'
  title       → 'Title'
  abstract    → 'Abstract'

[grouper] Normalising company names …
  Known companies     : 1,371
  Unknown/Independent :   268

[grouper] Embedding 1639 patents with PatentSBERTa …
  Model: AI-Growth-Lab/PatentSBERTa
  (downloads ~500 MB on first call)


/home/vasco/anaconda3/lib/python3.13/site-packages/pandas/core/algorithms.py:1743: UserWarning: rapidfuzz not installed — fuzzy company matching disabled. Only exact lookup table matches will be applied.
  return lib.map_infer(values, mapper, convert=convert)


...

Batches:   0%|          | 0/52 [00:00<?, ?it/s]


[grouper] Clustering by company group …
  AMSL Innovations                            4 patents → 0 prototype(s), 4 unclassified
  Aerhart                                     3 patents → 0 prototype(s), 3 unclassified
  AeroVironment                               3 patents → 0 prototype(s), 3 unclassified
  Aeronext                                    9 patents → 2 prototype(s), 1 unclassified
  Airbus                                     25 patents → 3 prototype(s), 12 unclassified
  Alphabet / X                                3 patents → 0 prototype(s), 3 unclassified
  Amazon                                      8 patents → 0 prototype(s), 8 unclassified
  Anduril                                     3 patents → 0 prototype(s), 3 unclassified
  Archer Aviation                            11 patents → 0 prototype(s), 11 unclassified
  Ascendance Flight Technologies              9 patents → 0 prototype(s), 9 unclassified
  Aurora Flight Sciences                     14 patents → 2 prototy

  Individual Inventor                       413 patents → 17 prototype(s), 328 unclassified
  Ishikawa Energy Research                    3 patents → 0 prototype(s), 3 unclassified
  JAXA                                        2 patents → 1 prototype(s), 0 unclassified
  Jetoptera                                   2 patents → 1 prototype(s), 0 unclassified
  Jiangxi Handun Duoyu                        2 patents → 1 prototype(s), 0 unclassified
  Joby Aviation                              20 patents → 2 prototype(s), 10 unclassified
  KAIST                                       3 patents → 0 prototype(s), 3 unclassified
  KARI                                       14 patents → 0 prototype(s), 14 unclassified
  Karem Aircraft                              9 patents → 0 prototype(s), 9 unclassified
  Kitty Hawk                                 30 patents → 2 prototype(s), 21 unclassified
  Kymatics                                    2 patents → 1 prototype(s), 0 unclassified
  Leonardo     


[grouper] Saved: /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/data/grouped_patents.csv
[grouper] Updated: /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/data/family_map.csv
[grouper] Saved: /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/data/dataset_summary.csv



[grouper] Company / Prototype summary:
  Company                                  Prototype         Count  Date range
  ────────────────────────────────────────────────────────────────────────────────
  AMSL Innovations                         Unclassified          4  2018-09 – 2025-04
  Aerhart                                  Unclassified          3  2021-07 – 2022-08
  AeroVironment                            Unclassified          3  2013-09 – 2017-06
  Aeronext                                 Prototype_A           5  2018-01 – 2021-03
  Aeronext                                 Prototype_B           3  2018-09 – 2020-09
  Aeronext                                 Unclassified          1  2015-05 – 2015-05
  Airbus                                   Prototype_A           3  2014-08 – 2016-05
  Airbus                                   Prototype_B           6  2018-02 – 2019-12
  Airbus                                   Prototype_C           4  2012-09 – 2022-11
  Airbus                

In [3]:
# ── Assign batch_id (300-400 per batch, clusters kept together) ───────────────
# Strategy: sort by display_order (clusters already grouped), walk forward
# and cut a new batch whenever we've collected >=MIN AND we just finished a cluster.
# MAX_BATCH_SIZE is a safety valve: some "clusters" are catch-all buckets (e.g.
# "Individual Inventor / Unclassified", 328 patents) rather than a real single
# company, and waiting for them to finish before cutting can swell a batch well
# past MIN (this produced a 606-patent batch before MAX_BATCH_SIZE existed). If a
# single cluster run itself would push a batch past MAX, force a cut mid-cluster —
# in practice this only ever splits these generic unclassified buckets, since every
# genuine (company, prototype) cluster here tops out around a dozen patents.

MIN_BATCH_SIZE = 300
MAX_BATCH_SIZE = 400

grouped_df = grouped_df.sort_values("display_order").reset_index(drop=True)

batch_ids    = []
batch_id     = 1
count        = 0
prev_cluster = None

for _, row in grouped_df.iterrows():
    cluster = (row["company_canonical"], row["prototype_label"])
    if cluster != prev_cluster:
        # Cut batch when we hit a new cluster AND have enough patents already
        if count >= MIN_BATCH_SIZE:
            batch_id += 1
            count = 0
    elif count >= MAX_BATCH_SIZE:
        # Same cluster still running but it's already swelled past MAX — force a
        # cut mid-cluster rather than let one giant bucket dominate a batch.
        batch_id += 1
        count = 0
    batch_ids.append(batch_id)
    count += 1
    prev_cluster = cluster

grouped_df["batch_id"] = batch_ids


In [4]:
# ── Create structured/ symlinks ────────────────────────────────────────────────
import os

matched_root    = Path(cfg["paths"]["matched"])
structured_root = Path(cfg["paths"]["structured"])

_PUB_VARIANTS = ["Publication Number", "Pub. No.", "Patent Number", "Record Number"]
pub_col = next((c for c in _PUB_VARIANTS if c in grouped_df.columns), None)
if pub_col is None:
    raise KeyError(f"No pub-number column found. Columns: {list(grouped_df.columns)}")

created  = 0
skipped  = 0
missing  = 0

for _, row in grouped_df.iterrows():
    patent_id = str(row[pub_col]).strip()
    company   = str(row["company_canonical"]).strip()
    prototype = str(row["prototype_label"]).strip()

    src_dir  = matched_root / patent_id
    link_dir = structured_root / company / prototype / patent_id

    if not src_dir.exists():
        missing += 1
        continue

    link_dir.parent.mkdir(parents=True, exist_ok=True)

    if link_dir.exists() or link_dir.is_symlink():
        skipped += 1
        continue

    os.symlink(src_dir.resolve(), link_dir)
    created += 1

print(f"Symlinks created : {created}")
print(f"Already existed  : {skipped}")
print(f"matched/ missing : {missing}  (run 00b first for these)")

Symlinks created : 0
Already existed  : 0
matched/ missing : 1639  (run 00b first for these)


In [5]:
# ── Write data/batches.xlsx ───────────────────────────────────────────────────
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font
from openpyxl.utils.dataframe import dataframe_to_rows

excel_index = load_patseer_excel(cfg["paths"]["patseer_excel"])
data_dir    = Path(cfg["paths"]["data"])
data_dir.mkdir(parents=True, exist_ok=True)
out_path    = data_dir / "batches.xlsx"

wb = Workbook()
# Remove default sheet
wb.remove(wb.active)

# ── Summary sheet ─────────────────────────────────────────────────────────────
summary_rows = []
for bid, grp in grouped_df.groupby("batch_id"):
    summary_rows.append({
        "batch_id":      int(bid),
        "patent_count":  len(grp),
        "companies":     ", ".join(sorted(grp["company_canonical"].unique())),
        "clusters":      ", ".join(sorted(grp["prototype_label"].unique())),
        "review_status": "pending",
    })

ws_sum = wb.create_sheet("Summary")
hdr_fill = PatternFill("solid", fgColor="1F497D")
hdr_font = Font(color="FFFFFF", bold=True)
sum_cols  = list(summary_rows[0].keys())
ws_sum.append(sum_cols)
for cell in ws_sum[1]:
    cell.fill = hdr_fill; cell.font = hdr_font
for r in summary_rows:
    ws_sum.append([r[c] for c in sum_cols])

# ── Per-batch sheets ──────────────────────────────────────────────────────────
BATCH_COLS = [
    "patent_id", "company_canonical", "prototype_label", "batch_id",
    "image_files", "description_of_drawings",
    "ai_label_json", "human_review_status", "notes",
]

for bid, grp in grouped_df.groupby("batch_id"):
    ws = wb.create_sheet(f"Batch_{int(bid):02d}")
    ws.append(BATCH_COLS)
    for cell in ws[1]:
        cell.fill = hdr_fill; cell.font = hdr_font

    for _, row in grp.iterrows():
        pid     = str(row[pub_col]).strip()
        meta    = excel_index.get(pid, {})
        img_dir = matched_root / pid
        imgs    = sorted(img_dir.glob("*.png")) if img_dir.exists() else []

        ws.append([
            pid,
            str(row["company_canonical"]),
            str(row["prototype_label"]),
            int(bid),
            ", ".join(f.name for f in imgs),
            meta.get("description_of_drawings", ""),
            "",   # ai_label_json — filled by 01_review
            "pending",
            "",
        ])

wb.save(out_path)
print(f"Saved {out_path}")
print(f"Sheets: {wb.sheetnames}")

/home/vasco/anaconda3/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


PatSeer Excel: 1639__dataset_08_06_26.xlsx  (1639 rows, 102 columns)
Columns:
  [  0] 'Record Number'
  [  1] 'Title'
  [  2] 'Abstract'
  [  3] 'Description of Drawings'
  [  4] 'CPC'
  [  5] 'PDF Link'
  [  6] 'Record Type'
  [  7] 'Publication/Issue Date'
  [  8] 'Filing/Application Date'
  [  9] 'Estimated Expiry Date'
  [ 10] 'EFAM Earliest Priority Date'
  [ 11] 'EFAM Earliest Publication Date'
  [ 12] 'Priority Details'
  [ 13] 'Priority Dates (All)'
  [ 14] 'Application No.'
  [ 15] 'Priority Country Code'
  [ 16] 'Priority Year'
  [ 17] 'Register Legal Status'
  [ 18] 'Register Legal Status Date'
  [ 19] 'Summary of Invention'
  [ 20] 'Designated States'
  [ 21] 'Active in Designated States'
  [ 22] 'Field of search'
  [ 23] 'Industry'
  [ 24] 'Tech Domain'
  [ 25] 'Tech Sub Domain'
  [ 26] 'Description'
  [ 27] 'Claims'
  [ 28] 'Number Of Claims'
  [ 29] 'No. of Independent Claims'
  [ 30] 'Independent Claims'
  [ 31] 'First Claim'
  [ 32] 'Advantages of Invention'
  [ 33] 'N

Saved /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/data/batches.xlsx
Sheets: ['Summary', 'Batch_01', 'Batch_02', 'Batch_03', 'Batch_04', 'Batch_05']
